In [6]:
# pip install sklearn
# pip install pythainlp
# pip install attacut

In [7]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import cosine_similarity
from pythainlp.tokenize import word_tokenize

In [10]:
class Tfidf : 
    def __init__(self, document, tokenizer=None, metric='cosine'):
        dict_metric = {'cosine': cosine_similarity, 'distance':euclidean_distances}
        self.metric = dict_metric.get(metric, None)
        if self.metric is None : 
            raise KeyError(f"metric: {dict_metric}")
        self.document = document
        self.tokenizer = tokenizer
        if self.tokenizer : 
            self.vectorizer = TfidfVectorizer(
                tokenizer=self.tokenizer, 
                token_pattern=None
            )
        else: 
            self.vectorizer = TfidfVectorizer()
        self.master_matrix = self.vectorizer.fit_transform(self.document)
        
    def search(self, query, top_k=5):
        query_result = []
        query_matrix = self.vectorizer.transform([query])
        distances = self.metric(query_matrix, self.master_matrix)
        top_k_indices = np.argsort(distances[0])[:top_k]
        for idx in top_k_indices : 
            query_result.append(
                {
                    'idx': idx, 
                    'query': self.document[idx], 
                    'score': distances[0][idx]
                }
            )
        return query_result

def attacut(txt):
    return word_tokenize(txt, engine='attacut', keep_whitespace=False)

In [11]:
documents = [
    "The cat sat on the mat.",
    "The dog barked loudly.",
    "The quick brown fox jumps over the lazy dog.",
    "A fox is a quick, cunning animal.",
    "Dogs are loyal and intelligent pets."
]

In [12]:
model = Tfidf(document=documents, tokenizer=attacut, metric='distance')

In [13]:
model.search(query='intelligent', top_k=3)

[{'idx': 4,
  'query': 'Dogs are loyal and intelligent pets.',
  'score': 1.0973964717619495},
 {'idx': 1, 'query': 'The dog barked loudly.', 'score': 1.414213562373095},
 {'idx': 0, 'query': 'The cat sat on the mat.', 'score': 1.4142135623730951}]